In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

# 노트북이 notebooks/ 안에 있으면 부모가 프로젝트 루트
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "cafe.db"
print("DB 존재?:", DB_PATH.exists(), "→", DB_PATH)

conn = sqlite3.connect(str(DB_PATH))

In [ ]:
# 어떤 테이블이 있나
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("테이블:", tables["name"].tolist())

# 각 테이블 컬럼 보기
for t in tables["name"]:
    cols = pd.read_sql(f"PRAGMA table_info({t})", conn)["name"].tolist()
    cnt = pd.read_sql(f"SELECT COUNT(*) c FROM {t}", conn)["c"][0]
    print(f"\n[{t}]  rows={cnt}")
    print("  cols:", cols)

In [ ]:
print("=== 카페 몇 개 ===")
display(pd.read_sql("SELECT * FROM cafes LIMIT 3", conn))

print("=== 한 카페의 키워드(긍/부정) ===")
display(pd.read_sql("""
    SELECT cafe_name, sentiment, keyword
    FROM cafe_keywords
    WHERE cafe_name = (SELECT cafe_name FROM cafe_keywords LIMIT 1)
""", conn))

print("=== 카페 수 / 키워드 어휘 크기 ===")
print(pd.read_sql("SELECT COUNT(DISTINCT name) n FROM cafes", conn))
print(pd.read_sql("SELECT sentiment, COUNT(DISTINCT keyword) n FROM cafe_keywords GROUP BY sentiment", conn))

In [ ]:
#셀 4 — 원천 테이블을 DataFrame으로 로드

cafes = pd.read_sql("SELECT name, address, status, review_summary FROM cafes", conn)
kw    = pd.read_sql("SELECT cafe_name, sentiment, keyword FROM cafe_keywords", conn)
menus = pd.read_sql("SELECT DISTINCT cafe_name, menu_name FROM cafe_menus", conn)

print("카페:", len(cafes), "| 키워드행:", len(kw), "| 메뉴행:", len(menus))
display(cafes.head(2))

In [ ]:
# 긍정/부정 키워드를 카페별 리스트로
pos = (kw[kw.sentiment == "긍정"]
       .groupby("cafe_name")["keyword"].apply(list).rename("pos_keywords"))
neg = (kw[kw.sentiment == "부정"]
       .groupby("cafe_name")["keyword"].apply(list).rename("neg_keywords"))
mn  = (menus.groupby("cafe_name")["menu_name"].apply(list).rename("menus"))

print("예시 — 한 카페의 긍정 키워드:")
print(pos.iloc[0][:10], "...")
print("\n부정 키워드:")
print(neg.iloc[0][:10] if len(neg) else "없음")

In [ ]:
# cafes(name) 기준으로 합치기
items = cafes.set_index("name").join([pos, neg, mn])

# 리스트 결측을 빈 리스트로
for col in ["pos_keywords", "neg_keywords", "menus"]:
    items[col] = items[col].apply(lambda v: v if isinstance(v, list) else [])

def build_profile(row):
    # 임베딩에 넣는 건 '의미' 신호만: 긍정키워드 + 리뷰요약 + 메뉴
    parts = []
    if row["pos_keywords"]:
        parts.append("특징: " + ", ".join(row["pos_keywords"]))
    if row["menus"]:
        parts.append("메뉴: " + ", ".join(row["menus"][:10]))
    if isinstance(row["review_summary"], str) and row["review_summary"].strip():
        parts.append("리뷰: " + row["review_summary"])
    return " | ".join(parts)

items["profile_text"] = items.apply(build_profile, axis=1)

print("총 아이템(카페):", len(items))
print("\n=== 한 카페 레코드 예시 ===")
ex = items.iloc[0]
print("이름:", ex.name)
print("pos:", ex["pos_keywords"][:8])
print("neg:", ex["neg_keywords"][:8], " ← 임베딩엔 안 넣음(감점 피처)")
print("profile_text:\n ", ex["profile_text"][:300])

In [ ]:
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv(ROOT / ".env")
MODEL_NAME = "east-right/bge-m3-cafe-finetuned"   # 네 하드네거티브 파인튜닝 모델(퍼블릭)

model = SentenceTransformer(MODEL_NAME, token=os.getenv("HUGGINGFACE_TOKEN_READ"))
print("로드 완료. 임베딩 차원:", model.get_sentence_embedding_dimension())

In [ ]:
import numpy as np

texts = items["profile_text"].tolist()
emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)  # (n, 1024)
print("임베딩 행렬:", emb.shape)

items["embedding"] = list(emb)   # 카페별 벡터 저장

In [ ]:
def search(query, k=5):
    q = model.encode(query, normalize_embeddings=True)
    sims = emb @ q                      # 정규화됨 → 내적 = 코사인 유사도
    idx = np.argsort(-sims)[:k]
    return [(items.index[i], round(float(sims[i]), 3)) for i in idx]

for query in ["조용하고 콘센트 있는 카공 카페", "디저트 맛있는 곳", "넓고 뷰 좋은 카페"]:
    print("Q:", query)
    for name, score in search(query):
        print(f"   {score}   {name}")
    print()

In [ ]:
items['profile_text']['카페 하나']

In [ ]:
items.loc['카페 하나']

In [ ]:
import json
q_path = ROOT / "data" / "valid_questions.json"
with open(q_path, encoding="utf-8") as f:
    raw_questions = json.load(f)

print("타입:", type(raw_questions))
print("개수:", len(raw_questions))
print("\n--- 첫 3개 ---")
for q in raw_questions[:3]:
    print(repr(q))

In [ ]:
# [셀 11] 질문셋 분포 분석 (어떻게 샘플링할지 결정용)
from collections import Counter

print("=== type 분포 ===")
print(Counter(q["type"] for q in raw_questions))

print("\n=== label 분포 ===")
print(Counter(q["label"] for q in raw_questions))

print("\n=== 질문당 키워드 개수 분포 ===")
print(Counter(len(q["keywords"]) for q in raw_questions))

print("\n=== 키워드 type(카테고리) 분포 ===")
kw_types = Counter(k["type"] for q in raw_questions for k in q["keywords"])
for t, c in kw_types.most_common():
    print(f"  {t:15s} {c}")

print("\n=== 고유 키워드 개수 ===")
uniq_kw = set(k["keyword"] for q in raw_questions for k in q["keywords"])
print(f"  {len(uniq_kw)}개")
print("  예시:", list(uniq_kw)[:20])

In [ ]:
# [셀 12] 질문 키워드 어휘  vs  카페 데이터 키워드 어휘 (호환성 게이트)

# 카페 쪽 긍정 키워드 전체 어휘
cafe_vocab = set()
for kws in items["pos_keywords"]:
    cafe_vocab.update(kws)
print(f"카페 pos_keywords 고유 어휘: {len(cafe_vocab)}개")

# 질문 쪽 키워드 전체 어휘
q_vocab = set(k["keyword"] for q in raw_questions for k in q["keywords"])
print(f"질문 키워드 고유 어휘:      {len(q_vocab)}개")

# 교집합 / 질문에만 있는 것(= 데이터로 답 불
overlap = q_vocab & cafe_vocab
only_q  = q_vocab - cafe_vocab
print(f"\n겹치는 키워드:        {len(overlap)}개  ({len(overlap)/len(q_vocab)*100:.0f}% of 질문어휘)")
print(f"질문에만 있는 키워드:  {len(only_q)}개  ← 이 키워드만 가진 질문은 데이터로 답 불가")

print("\n--- 겹치는 키워드 예시 ---")
print(sorted(overlap)[:30])
print("\n--- 질문에만 있는 키워드(고아) ---")
print(sorted(only_q)[:30])

In [ ]:
# [셀 13] 질문셋 → DataFrame + 답가능 필터 + 고유 요구 기준 ~150개 추출
import pandas as pd

rows = []
for q in raw_questions:
    kws = [k["keyword"] for k in q["keywords"]]
    rows.append({
        "question": q["question"],
        "type": q["type"],
        "n_kw": len(kws),
        "kw_list": kws,
        "kw_set": frozenset(kws),
        "kw_types": tuple(sorted(set(k["type"] for k in q["keywords"]))),
        "answerable": all(k in cafe_vocab for k in kws),   # 모든 키워드가 카페 어휘에 있나
    })
df_q = pd.DataFrame(rows)

print(f"전체 질문: {len(df_q)}")
print(f"답가능(모든 키워드 매칭): {df_q['answerable'].sum()}")
print(f"고유 키워드조합(요구) 수: {df_q['kw_set'].nunique()}")
print(f"  - single 요구: {df_q[df_q.type=='single']['kw_set'].nunique()}")
print(f"  - multi 요구:  {df_q[df_q.type=='multi']['kw_set'].nunique()}")

# 답가능한 것만, 고유 요구당 1개 대표 질문 추출
ans = df_q[df_q.answerable].copy()
rep = ans.groupby("kw_set", group_keys=False).sample(1, random_state=42)
print(f"\n고유 요구 대표질문 풀: {len(rep)}  (single {sum(rep.type=='single')}, multi {sum(rep.type=='multi')})")

# 층화추출: single 80 + multi 70 ≈ 150 (복합질문 비중↑ = 랭킹 변별 중요)
n_single = min(80, sum(rep.type=="single"))
n_multi  = min(70, sum(rep.type=="multi"))
QUERY_SET = pd.concat([
    rep[rep.type=="single"].sample(n_single, random_state=42),
    rep[rep.type=="multi"].sample(n_multi,  random_state=42),
]).reset_index(drop=True)
QUERY_SET.insert(0, "qid", [f"q{i:03d}" for i in range(len(QUERY_SET))])

print(f"\n=== 최종 QUERY_SET: {len(QUERY_SET)}개 ===")
print(QUERY_SET["type"].value_counts().to_dict())
print(QUERY_SET["n_kw"].value_counts().sort_index().to_dict())
QUERY_SET[["qid","question","type","kw_list"]].head(10)

In [ ]:
# [셀 14] 후보 풀링 — 질문마다 카페 ~20개 추리기
import numpy as np

K_EMB  = 15   # 임베딩 유사도 상위
K_KW   = 15   # 키워드 겹침 상위
N_RAND = 3    # 랜덤(쉬운 오답) 보강
items = items.reset_index()
cafe_names = items["name"].tolist()      # emb 행 순서와 일치
cafe_pos   = items["pos_keywords"].tolist()
print(f"전체 카페 수: {len(cafe_names)}")

rng = np.random.default_rng(42)

def pool_candidates(query_text, query_kws):
    # 1) 임베딩(의미) 상위 — 패러프레이즈/유사개념까지 포착
    q = model.encode([query_text], normalize_embeddings=True)[0]
    emb_top = np.argsort(-(emb @ q))[:K_EMB]
    # 2) 키워드 겹침(정확) 상위 — 요구 키워드 직접 보유
    overlaps = np.array([len(set(query_kws) & set(kws)) for kws in cafe_pos])
    kw_top = np.argsort(-overlaps)[:K_KW]
    # 3) 합집합 + 랜덤(확실한 오답 보장)
    idx = set(emb_top) | set(kw_top) | set(rng.choice(len(cafe_names), N_RAND, replace=False))
    return sorted(idx)

pools = {row["qid"]: pool_candidates(row["question"], row["kw_list"])
         for _, row in QUERY_SET.iterrows()}

sizes = [len(v) for v in pools.values()]
print(f"질문당 후보: 평균 {np.mean(sizes):.1f}, 최소 {min(sizes)}, 최대 {max(sizes)}")
print(f"전체 (질문,카페) 쌍: {sum(sizes)}개  ← 다음 단계서 이만큼 정답 매김")

# 샘플 확인
ex = QUERY_SET.iloc[0]
print(f"\n예시: {ex['question']}  (요구: {ex['kw_list']})")
for i in pools[ex["qid"]][:8]:
    ov = set(ex["kw_list"]) & set(cafe_pos[i])
    print(f"  {cafe_names[i]:18s} 겹침={len(ov)} {sorted(ov)}")

In [ ]:
# [셀 14b] 진단 — '소음' 요구가 실제로 매칭되는지 (인덱스순 말고 겹침순으로)
target_kw = "소음"

# 1) 전체 카페 중 '소음'을 pos_keyword로 가진 곳이 있나?
have = [(cafe_names[i], cafe_pos[i]) for i in range(len(cafe_names)) if target_kw in cafe_pos[i]]
print(f"'{target_kw}' 가진 카페: {len(have)}개")
for name, kws in have[:5]:
    print(f"  {name}: {sorted(kws)[:10]}")

# 2) 이 질문 풀을 '겹침 높은 순'으로 다시 정렬
ex = QUERY_SET.iloc[0]
cand = sorted(pools[ex["qid"]],
              key=lambda i: len(set(ex["kw_list"]) & set(cafe_pos[i])),
              reverse=True)
print(f"\n질문: {ex['question']}  (요구 {ex['kw_list']}) — 겹침 높은 순:")
for i in cand[:8]:
    ov = set(ex["kw_list"]) & set(cafe_pos[i])
    print(f"  {cafe_names[i]:18s} 겹침={len(ov)} {sorted(ov)}")

In [ ]:
# [셀 14c] '조용함' 개념은 여러 키워드로 흩어져 있다 — 정확매칭의 한계 확인
literal = [cafe_names[i] for i in range(len(cafe_pos)) if "소음" in cafe_pos[i]]
concept = [cafe_names[i] for i in range(len(cafe_pos))
           if any(("조용" in k or "소음" in k or "답답하지 않" in k) for k in cafe_pos[i])]
print(f"'소음' 정확히 가진 카페:      {len(literal)}개")
print(f"'조용함' 개념 가진 카페:      {len(concept)}개")
print("개념 카페 예시:", concept[:10])

In [ ]:
# [셀 15] LLM-as-judge — 1개 질문으로 테스트 (전체 돌리기 전 검증)
import os, json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
client = OpenAI()                       # .env의 OPENAI_API_KEY 사용

cafe_neg  = items["neg_keywords"].tolist()
cafe_menu = items["menus"].tolist()

def cafe_brief(i):
    pos  = ", ".join(cafe_pos[i][:12])
    neg  = ", ".join(cafe_neg[i][:6])  if cafe_neg[i]  else "없음"
    menu = ", ".join(cafe_menu[i][:6]) if cafe_menu[i] else "없음"
    return f"[{i}] {cafe_names[i]} | 특징: {pos} | 단점: {neg} | 메뉴: {menu}"

RUBRIC = """당신은 카페 추천 평가자입니다. 사용자 질문에 각 카페가 얼마나 맞는지 0~3으로 채점하세요.
3 = 질문의 모든 요구 충족 / 2 = 핵심 충족(일부 부족) / 1 = 부분적 관련 / 0 = 무관하거나 요구에 반함.
중요: 키워드가 글자 그대로 없어도 의미가 맞으면 점수를 주세요(예: '조용한' 요구에 '분위기_조용하다' 카페는 3점). '단점'이 요구에 반하면 감점하세요."""

def judge_query(question, cand_idx):
    cafes = "\n".join(cafe_brief(i) for i in cand_idx)
    prompt = (f'질문: "{question}"\n\n후보 카페:\n{cafes}\n\n'
              '각 카페 번호에 0~3 점수를 매겨 JSON으로만 답하세요. '
              '형식: {"scores": {"0": 3, "5": 1, ...}}')
    resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role":"system","content":RUBRIC},
                  {"role":"user","content":prompt}],
        response_format={"type":"json_object"}, temperature=0,
    )
    return json.loads(resp.choices[0].message.content)["scores"]

# --- 1개만 테스트 ---
ex = QUERY_SET.iloc[0]; cand = pools[ex["qid"]]
scores = judge_query(ex["question"], cand)
print(f"질문: {ex['question']}  (요구: {ex['kw_list']})\n")
for i in sorted(cand, key=lambda i: -int(scores.get(str(i), 0))):
    ov = sorted(set(ex["kw_list"]) & set(cafe_pos[i]))
    print(f"  {int(scores.get(str(i),0))}점  {cafe_names[i]:18s} 겹침{ov}")

In [ ]:
# [셀 16] LLM-judge → qrels (캐시 있으면 재호출 안 함)
import time, json
qrels_path = ROOT/"data"/"qrels.csv"

if qrels_path.exists():
    qrels = pd.read_csv(qrels_path)
    print(f"✅ 캐시 로드: {qrels_path.name} ({len(qrels)}쌍) — LLM 재호출 안 함")
    print("관련도 분포:", qrels["relevance"].value_counts().sort_index().to_dict())
else:
    print("캐시 없음 → LLM-judge 150콜 시작 (몇 분 + API 비용)")
    qrels_rows, errors = [], []
    for n, (_, row) in enumerate(QUERY_SET.iterrows()):
        qid, cand = row["qid"], pools[row["qid"]]
        try:
            scores = judge_query(row["question"], cand)
        except Exception:                            # 1회 재시도
            time.sleep(2)
            try:
                scores = judge_query(row["question"], cand)
            except Exception as e2:
                errors.append((qid, str(e2))); continue
        for i in cand:
            qrels_rows.append({"qid": qid, "cafe_idx": int(i),
                               "cafe_name": cafe_names[i],
                               "relevance": int(scores.get(str(i), 0))})
        if (n+1) % 20 == 0:
            print(f"  {n+1}/{len(QUERY_SET)} ...")

    qrels = pd.DataFrame(qrels_rows)
    print(f"\n총 {len(qrels)}쌍, 에러 {len(errors)}건")
    print("관련도 분포:", qrels["relevance"].value_counts().sort_index().to_dict())

    # 저장 (다음 실행부턴 위 캐시 분기로 빠짐)
    qrels.to_csv(qrels_path, index=False, encoding="utf-8")
    qs = QUERY_SET[["qid","question","type","n_kw","kw_list"]].copy()
    qs["kw_list"] = qs["kw_list"].apply(list)
    qs.to_json(ROOT/"data"/"query_set.json", orient="records", force_ascii=False)
    json.dump({q:[int(i) for i in v] for q,v in pools.items()},
              open(ROOT/"data"/"pools.json","w"), ensure_ascii=False)
    print("저장 완료: qrels.csv / query_set.json / pools.json")

In [ ]:
# [셀 17] 피처 계산 → 학습표 완성
q_emb  = {r["qid"]: model.encode([r["question"]], normalize_embeddings=True)[0] for _, r in QUERY_SET.iterrows()}
qid2kws = {r["qid"]: set(r["kw_list"]) for _, r in QUERY_SET.iterrows()}
qid2nkw = {r["qid"]: r["n_kw"]         for _, r in QUERY_SET.iterrows()}

rows = []
for _, r in qrels.iterrows():
    qid, i = r["qid"], r["cafe_idx"]
    qk = qid2kws[qid]
    pos, neg, menu = set(cafe_pos[i]), set(cafe_neg[i]), set(cafe_menu[i])
    ov = len(qk & pos)
    rows.append({
        "qid": qid, "cafe_idx": i, "relevance": r["relevance"],
        "emb_cos":          float(emb[i] @ q_emb[qid]),   # 의미 유사도
        "kw_overlap":       ov,                           # 정확 키워드 겹침
        "kw_overlap_ratio": ov / max(len(qk), 1),
        "menu_overlap":     len(qk & menu),               # 메뉴 매칭
        "n_pos_kw":         len(pos),                      # 카페 긍정수
        "n_neg_kw":         len(neg),                      # 카페 단점수
        "pos_neg_ratio":    len(pos)/(len(pos)+len(neg)+1),# 감성(극성) 프록시
        "n_query_kw":       qid2nkw[qid],                 # 질문 조건수
    })
data = pd.DataFrame(rows)
print("학습표:", data.shape)
data.head()

In [ ]:
# [셀 18] 질문 단위 train/val/test 분리 (누수 방지)
import numpy as np
qids = QUERY_SET["qid"].tolist()
np.random.default_rng(42).shuffle(qids)
n = len(qids)
train_q, val_q, test_q = set(qids[:int(n*.7)]), set(qids[int(n*.7):int(n*.85)]), set(qids[int(n*.85):])
print(f"train {len(train_q)} / val {len(val_q)} / test {len(test_q)} 질문")

FEATS = ["emb_cos","kw_overlap","kw_overlap_ratio","menu_overlap",
         "n_pos_kw","n_neg_kw","pos_neg_ratio","n_query_kw"]

def make_xy(split):
    d = data[data.qid.isin(split)].sort_values("qid").reset_index(drop=True)
    g = d.groupby("qid", sort=True).size().to_numpy()   # 질문별 후보 수(그룹)
    return d[FEATS].to_numpy(), d["relevance"].to_numpy(), g, d

X_tr,y_tr,g_tr,_   = make_xy(train_q)
X_va,y_va,g_va,_   = make_xy(val_q)
X_te,y_te,g_te,d_te= make_xy(test_q)
print("train쌍", len(y_tr), "/ 질문", len(g_tr))

In [ ]:
# [셀 19] 베이스라인 NDCG (학습 안 한 단순 정렬)
from sklearn.metrics import ndcg_score

def eval_ndcg(d, score_col, k=5):
    vals = []
    for _, grp in d.groupby("qid"):
        if len(grp) < 2 or grp["relevance"].max() == 0:   # 정답 없는 질문 제외
            continue
        vals.append(ndcg_score(grp["relevance"].values.reshape(1,-1),
                               grp[score_col].values.reshape(1,-1), k=k))
    return float(np.mean(vals))

print("=== Test NDCG@5 (베이스라인) ===")
print("  임베딩만 정렬     :", round(eval_ndcg(d_te, "emb_cos"), 4))
print("  키워드겹침만 정렬 :", round(eval_ndcg(d_te, "kw_overlap"), 4))

In [ ]:
# [셀 20] LambdaMART 학습 (LightGBM lambdarank)
import lightgbm as lgb
train_set = lgb.Dataset(X_tr, label=y_tr, group=g_tr)
val_set   = lgb.Dataset(X_va, label=y_va, group=g_va, reference=train_set)
params = {"objective":"lambdarank", "metric":"ndcg", "ndcg_eval_at":[1,3,5],
          "learning_rate":0.05, "num_leaves":15, "min_data_in_leaf":20, "verbose":-1}
ranker = lgb.train(params, train_set, num_boost_round=300, valid_sets=[val_set],
                   callbacks=[lgb.early_stopping(30), lgb.log_evaluation(20)])

In [ ]:
# [셀 21] 학습 모델 평가 — 베이스라인 vs LambdaMART
d_te = d_te.copy()
d_te["pred"] = ranker.predict(X_te)
print("=== Test NDCG@5 ===")
print("  임베딩만        :", round(eval_ndcg(d_te,"emb_cos"),4))
print("  키워드겹침만    :", round(eval_ndcg(d_te,"kw_overlap"),4))
print("  ★ LambdaMART   :", round(eval_ndcg(d_te,"pred"),4))

In [ ]:
# [셀 22] SHAP — 어떤 피처가 추천을 결정했나 (faithful 설명)
import shap
explainer = shap.TreeExplainer(ranker)
sv = explainer.shap_values(X_te)
shap.summary_plot(sv, X_te, feature_names=FEATS, show=True)

# 한 추천에 대한 근거 예시
ex_qid = d_te["qid"].iloc[0]
g = d_te[d_te.qid==ex_qid].copy()
top = g.sort_values("pred", ascending=False).iloc[0]
ti = g.index.get_loc(top.name)
print(f"\n질문: {QUERY_SET.set_index('qid').loc[ex_qid,'question']}")
print(f"1위 추천: {cafe_names[int(top['cafe_idx'])]}  (관련도 정답 {int(top['relevance'])})")
print("근거(피처 기여):")
for f, val in sorted(zip(FEATS, sv[ti]), key=lambda x:-abs(x[1]))[:5]:
    print(f"  {f:18s} 기여 {val:+.3f}")

In [ ]:
print("총 아이템(카페):", len(items))
print("\n=== 한 카페 레코드 예시 ===")
ex = items[items['name'] == '이디야커피(신림패션문화점)']
print("이름:", ex.name)
print("pos:", ex["pos_keywords"][:])
print("neg:", ex["neg_keywords"][:8], " ← 임베딩엔 안 넣음(감점 피처)")
print("profile_text:\n ", ex["profile_text"][:300])

In [ ]:
ex["pos_keywords"]

In [ ]:
items[items['name'] == '이디야커피(신림패션문화점)']["pos_keywords"].to_list()

In [ ]:


import json

# 파일 열기 (읽기 모드)
with open('./data/ner_result_v2.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

In [ ]:
data

In [ ]:
df = pd.read_csv("./data/shinline_cafe_reviews_test.csv")

In [ ]:
df

In [ ]:
import pandas as pd

# data: {place_id: [entity_dict, entity_dict, ...]}
# df: 사업장명, place_id, 리뷰내용, 작성일

# 1. data를 평탄화 (flatten)
rows = []
for place_id, entities in data.items():
    for ent in entities:
        rows.append({
            'place_id': place_id,
            'entity': ent.get('entity'),
            'category': ent.get('category'),
            'type': ent.get('type'),
            'sentiment': ent.get('sentiment'),
            'descriptor': ent.get('descriptor'),
            '리뷰내용': ent.get('source_review'),  # df의 리뷰내용과 매칭용
        })

keyword_df = pd.DataFrame(rows)

# 2. place_id 타입 통일 (df가 int, data 키가 str일 수 있음 — 주의)
df['place_id'] = df['place_id'].astype(str)
keyword_df['place_id'] = keyword_df['place_id'].astype(str)

# 3. place_id + 리뷰내용 기준으로 merge
merged = pd.merge(
    df,
    keyword_df,
    on=['place_id', '리뷰내용'],
    how='inner'  # 키워드 추출된 리뷰만 보고 싶으면 inner, 전체 리뷰 다 보고 싶으면 left
)

print(merged.shape)
merged.head()

In [ ]:
merged[merged['사업장명'] == '이디야커피(신림패션문화점)'][0:10]

In [ ]:
# [진단] 이디야(패션문화점) '뷰' 강점이 진짜인가 — 원본 리뷰 증거
import pandas as pd
rv = pd.read_csv(ROOT/"data"/"shinline_cafe_reviews_test.csv")
print("리뷰 컬럼:", list(rv.columns))     # 매장/리뷰 컬럼명 확인(아니면 직접 지정)

name_col = next(c for c in rv.columns if any(k in c.lower() for k in ['사업장명', 'place_id', '리뷰내용', '작성일']))
text_col = next(c for c in rv.columns if any(k in c.lower() for k in ["review","리뷰","content","text","내용"]))

sub = rv[rv[name_col].astype(str).str.contains("이디야", na=False)
         & rv[name_col].astype(str).str.contains("패션", na=False)]
print(f"\n이디야(패션문화점) 총 리뷰: {len(sub)}")
view = sub[sub[text_col].astype(str).str.contains("뷰|전망|경치|view", na=False)]
print(f"'뷰/전망' 언급: {len(view)}건")
for t in view[text_col].head(5):
    print(" -", str(t)[:120])